# Train TRRUST Classifiers

Hyperparameter tuning, optional sample subsetting, and 5-fold cross-validation for
binary (Relationship vs None) and ternary (Activation vs Repression vs None) TRRUST
classifiers.

Pipeline:

1. **Load** gene embeddings + TRRUST data.
2. **(Optional)** Restrict training data to a gene subset from a file.
3. **(Optional)** Write the current gene universe to a file for cross-model comparisons.
4. **Hyperparameter sweep** — 5 LRs × 4 epoch counts for binary + ternary; plots,
   per-config classification reports, a summary CSV, and an F1 heatmap are written
   under `REPORTS_DIR`.
5. **5-fold stratified cross-validation** for binary + ternary at a user-chosen
   LR / epoch count (inspect the sweep outputs, then set `*_LR` / `*_EPOCHS`).
6. **Save CV results** as a pickle for downstream analysis.

This notebook is model-agnostic — point `EMBEDDINGS_PATH` at any h5ad file produced
by `src/scgpt/encode.py` or `src/geneformer/encode.py`.


In [1]:
import itertools
import pickle
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report

from src.trrust import (
    BINARY_LABEL_NAMES,
    BINARY_LABELS,
    TERNARY_LABEL_NAMES,
    TERNARY_LABELS,
    cross_validate,
    filter_data_by_genes,
    load_binary_trrust_data,
    load_gene_embeddings,
    load_ternary_trrust_data,
    prepare_train_test_split,
    train_classifier,
)


## Configuration

Change `EMBEDDINGS_PATH` to swap foundation models, and point `REPORTS_DIR` /
`CV_RESULTS_PATH` at model-specific output locations. Set `GENE_SUBSET_PATH` to
restrict training to pairs where both TF and target appear in a gene-list file
(see `data/scgpt_binary_genes.txt`).

`LR_GRID` and `EPOCH_GRID` define the hyperparameter sweep. After inspecting the
sweep outputs, fill in `BINARY_LR` / `BINARY_EPOCHS` / `TERNARY_LR` / `TERNARY_EPOCHS`
before running cross-validation.


In [ ]:
EMBEDDINGS_PATH = Path("../data/embeddings/scgpt_human_cd20.h5ad")
TRRUST_PATH = Path("../data/trrust_rawdata.human.tsv")
REPORTS_DIR = Path("../reports/scgpt/hp_tuning_cd20")
CV_RESULTS_PATH = Path("../data/scgpt_cv_results_cd20.pkl")
GENE_SUBSET_PATH: Path | None = None  # e.g. Path("../data/scgpt_binary_genes.txt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

LR_GRID = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
EPOCH_GRID = [6, 12, 20, 50]

# Set these after inspecting the hyperparameter sweep output.
TERNARY_LR = 1e-5
TERNARY_EPOCHS = 20
N_FOLDS = 5

print(f"Device: {DEVICE}")
print(f"Embeddings: {EMBEDDINGS_PATH}")
print(f"TRRUST: {TRRUST_PATH}")
print(f"Reports dir: {REPORTS_DIR}")
print(f"CV results: {CV_RESULTS_PATH}")
print(f"Gene subset: {GENE_SUBSET_PATH}")


Device: cuda
Embeddings: ../data/embeddings/scgpt_human_cd20.h5ad
TRRUST: ../data/trrust_rawdata.human.tsv
Reports dir: ../reports/scgpt/hp_tuning_cd20
CV results: ../data/scgpt_cv_results_cd20.pkl
Gene subset: None


## Load embeddings and TRRUST data

In [5]:
gene_embeddings = load_gene_embeddings(EMBEDDINGS_PATH)
embsize = next(iter(gene_embeddings.values())).shape[0]

binary_data = load_binary_trrust_data(TRRUST_PATH, gene_embeddings)
ternary_data = load_ternary_trrust_data(TRRUST_PATH, gene_embeddings)

print(f"Gene embeddings: {len(gene_embeddings)} genes, {embsize}d")
print(f"Binary samples: {len(binary_data.records)}")
print(f"Ternary samples: {len(ternary_data.records)}")


Gene embeddings: 10856 genes, 512d
Binary samples: 8100
Ternary samples: 3147


## (Optional) Restrict training data to a gene subset

If `GENE_SUBSET_PATH` is set, keep only TRRUST records where both TF and target are
listed in the file (one gene symbol per line). A no-op when `GENE_SUBSET_PATH is None`.


In [ ]:
if GENE_SUBSET_PATH is not None:
    subset_genes = {
        line.strip()
        for line in Path(GENE_SUBSET_PATH).read_text().splitlines()
        if line.strip()
    }
    print(f"Subset file: {GENE_SUBSET_PATH} ({len(subset_genes)} genes)")

    before_binary = len(binary_data.records)
    before_ternary = len(ternary_data.records)
    binary_data = filter_data_by_genes(binary_data, subset_genes)
    ternary_data = filter_data_by_genes(ternary_data, subset_genes)
    print(f"Binary records: {before_binary} -> {len(binary_data.records)}")
    print(f"Ternary records: {before_ternary} -> {len(ternary_data.records)}")
else:
    print("GENE_SUBSET_PATH is None — using all records.")


## (Optional) Save current gene lists for future cross-model runs

Write the union of TFs + targets from the current `binary_data` / `ternary_data` to
two `.txt` files so another run (e.g. a different embedding model) can be restricted
to the same gene universe via `GENE_SUBSET_PATH` above. Edit the output paths for
the model you are running.


In [ ]:
BINARY_GENE_FILE = Path("../data/scgpt_binary_genes.txt")
TERNARY_GENE_FILE = Path("../data/scgpt_ternary_genes.txt")

binary_genes = sorted({g for r in binary_data.records for g in (r.tf, r.target)})
BINARY_GENE_FILE.write_text("\n".join(binary_genes) + "\n")
print(f"Saved {len(binary_genes)} binary genes to {BINARY_GENE_FILE}")

ternary_genes = sorted({g for r in ternary_data.records for g in (r.tf, r.target)})
TERNARY_GENE_FILE.write_text("\n".join(ternary_genes) + "\n")
print(f"Saved {len(ternary_genes)} ternary genes to {TERNARY_GENE_FILE}")


## Hyperparameter sweep helpers

Each config trains on the same 90/10 stratified split (seed=42) so results are
comparable across the grid. For every `(lr, epochs)` combination we save the loss
curve, the text classification report, and a row in `summary.csv`. After the loop,
an F1 heatmap over LR × epochs is saved alongside the per-config outputs.


In [6]:
def _save_loss_plot(result, lr, epochs, title_prefix, out_path):
    plt.figure(figsize=(8, 4))
    xs = range(1, epochs + 1)
    plt.plot(xs, result.train_losses, label="Train")
    plt.plot(xs, result.test_losses, label="Test")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} — lr={lr:.0e}, epochs={epochs}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def _save_text_report(result, label_names, out_path):
    preds = result.gene_predictions
    name_to_label = {name: idx for idx, name in label_names.items()}
    true_ids = preds["true_relationship"].map(name_to_label).to_numpy()
    pred_ids = preds["predicted_relationship"].map(name_to_label).to_numpy()
    target_names = [label_names[i] for i in range(len(label_names))]
    text = classification_report(
        true_ids, pred_ids, target_names=target_names, zero_division=0
    )
    out_path.write_text(text)


def _summary_row(lr, epochs, result, label_names):
    report = result.classification_report
    row = {"lr": lr, "epochs": epochs, "accuracy": report["accuracy"]}
    for agg in ("macro avg", "weighted avg"):
        for metric in ("precision", "recall", "f1-score"):
            row[f"{agg}_{metric}"] = report[agg][metric]
    for i in range(len(label_names)):
        name = label_names[i]
        row[f"{name}_precision"] = report[name]["precision"]
        row[f"{name}_recall"] = report[name]["recall"]
        row[f"{name}_f1"] = report[name]["f1-score"]
    return row


def _save_f1_heatmap(summary_df, title, out_path):
    pivot = summary_df.pivot(
        index="lr", columns="epochs", values="macro avg_f1-score"
    ).sort_index(ascending=False)
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.0e}" for v in pivot.index])
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Learning rate")
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center",
                    color="white" if pivot.values[i, j] < pivot.values.mean() else "black")
    fig.colorbar(im, ax=ax, label="macro F1")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def run_hp_sweep(data, *, label_map, label_names, use_class_weights, out_dir, title_prefix):
    out_dir.mkdir(parents=True, exist_ok=True)
    train_ds, test_ds, test_meta = prepare_train_test_split(data, test_size=0.1, seed=42)

    rows = []
    for lr, epochs in itertools.product(LR_GRID, EPOCH_GRID):
        result = train_classifier(
            train_ds,
            test_ds,
            test_meta,
            embsize=embsize,
            label_map=label_map,
            lr=lr,
            epochs=epochs,
            batch_size=BATCH_SIZE,
            use_class_weights=use_class_weights,
            device=DEVICE,
        )
        tag = f"lr{lr:.0e}_e{epochs}"
        _save_loss_plot(result, lr, epochs, title_prefix, out_dir / f"loss_{tag}.png")
        _save_text_report(result, label_names, out_dir / f"report_{tag}.txt")
        rows.append(_summary_row(lr, epochs, result, label_names))
        print(f"  {tag}: acc={result.classification_report['accuracy']:.3f}, "
              f"macro F1={result.classification_report['macro avg']['f1-score']:.3f}")

    summary = pd.DataFrame(rows)
    summary.to_csv(out_dir / "summary.csv", index=False)
    _save_f1_heatmap(summary, f"{title_prefix} macro F1", out_dir / "f1_heatmap.png")
    return summary


## Hyperparameter sweep — binary classifier

20 configs (5 LRs × 4 epoch counts), no class weights. Outputs land in
`REPORTS_DIR/binary/`.


In [7]:
binary_summary = run_hp_sweep(
    binary_data,
    label_map=BINARY_LABELS,
    label_names=BINARY_LABEL_NAMES,
    use_class_weights=False,
    out_dir=REPORTS_DIR / "binary",
    title_prefix="Binary",
)
binary_summary.sort_values("macro avg_f1-score", ascending=False).head()


  lr1e-02_e6: acc=0.653, macro F1=0.652
  lr1e-02_e12: acc=0.686, macro F1=0.682
  lr1e-02_e20: acc=0.638, macro F1=0.605
  lr1e-02_e50: acc=0.500, macro F1=0.333
  lr1e-03_e6: acc=0.625, macro F1=0.612
  lr1e-03_e12: acc=0.501, macro F1=0.336
  lr1e-03_e20: acc=0.702, macro F1=0.699
  lr1e-03_e50: acc=0.626, macro F1=0.574
  lr1e-04_e6: acc=0.604, macro F1=0.571
  lr1e-04_e12: acc=0.523, macro F1=0.389
  lr1e-04_e20: acc=0.679, macro F1=0.669
  lr1e-04_e50: acc=0.542, macro F1=0.420
  lr1e-05_e6: acc=0.577, macro F1=0.574
  lr1e-05_e12: acc=0.594, macro F1=0.565
  lr1e-05_e20: acc=0.640, macro F1=0.634
  lr1e-05_e50: acc=0.679, macro F1=0.677
  lr1e-06_e6: acc=0.537, macro F1=0.535
  lr1e-06_e12: acc=0.525, macro F1=0.513
  lr1e-06_e20: acc=0.541, macro F1=0.541
  lr1e-06_e50: acc=0.599, macro F1=0.596


,lr,epochs,accuracy,macro avg_precision,macro avg_recall,macro avg_f1-score,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,None_precision,None_recall,None_f1,Relationship_precision,Relationship_recall,Relationship_f1
6,0.00100,20,0.702469,0.711346,0.702469,0.699312,0.711346,0.702469,0.699312,0.668033,0.804938,0.730123,0.754658,0.600000,0.668501
1,0.01000,12,0.686420,0.696563,0.686420,0.682322,0.696563,0.686420,0.682322,0.651911,0.800000,0.718404,0.741214,0.572840,0.646240
15,0.00001,50,0.679012,0.682761,0.679012,0.677358,0.682761,0.679012,0.677358,0.708934,0.607407,0.654255,0.656587,0.750617,0.700461
10,0.00010,20,0.679012,0.704918,0.679012,0.668536,0.704918,0.679012,0.668536,0.632058,0.856790,0.727463,0.777778,0.501235,0.609610
0,0.01000,6,0.653086,0.655361,0.653086,0.651812,0.655361,0.653086,0.651812,0.674157,0.592593,0.630749,0.636564,0.713580,0.672875


## Hyperparameter sweep — ternary classifier

20 configs with inverse-frequency class weights. Outputs land in
`REPORTS_DIR/ternary/`.


In [8]:
ternary_summary = run_hp_sweep(
    ternary_data,
    label_map=TERNARY_LABELS,
    label_names=TERNARY_LABEL_NAMES,
    use_class_weights=True,
    out_dir=REPORTS_DIR / "ternary",
    title_prefix="Ternary",
)
ternary_summary.sort_values("macro avg_f1-score", ascending=False).head()


  lr1e-02_e6: acc=0.263, macro F1=0.139
  lr1e-02_e12: acc=0.273, macro F1=0.159
  lr1e-02_e20: acc=0.267, macro F1=0.146
  lr1e-02_e50: acc=0.527, macro F1=0.516
  lr1e-03_e6: acc=0.263, macro F1=0.139
  lr1e-03_e12: acc=0.295, macro F1=0.215
  lr1e-03_e20: acc=0.403, macro F1=0.192
  lr1e-03_e50: acc=0.454, macro F1=0.373
  lr1e-04_e6: acc=0.267, macro F1=0.157
  lr1e-04_e12: acc=0.292, macro F1=0.212
  lr1e-04_e20: acc=0.368, macro F1=0.311
  lr1e-04_e50: acc=0.460, macro F1=0.372
  lr1e-05_e6: acc=0.359, macro F1=0.321
  lr1e-05_e12: acc=0.397, macro F1=0.291
  lr1e-05_e20: acc=0.292, macro F1=0.205
  lr1e-05_e50: acc=0.457, macro F1=0.359
  lr1e-06_e6: acc=0.276, macro F1=0.270
  lr1e-06_e12: acc=0.286, macro F1=0.285
  lr1e-06_e20: acc=0.359, macro F1=0.339
  lr1e-06_e50: acc=0.343, macro F1=0.320


,lr,epochs,accuracy,macro avg_precision,macro avg_recall,macro avg_f1-score,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,Activation_precision,Activation_recall,Activation_f1,Repression_precision,Repression_recall,Repression_f1,None_precision,None_recall,None_f1
3,0.010000,50,0.526984,0.546439,0.548160,0.515840,0.561294,0.526984,0.508899,0.655172,0.299213,0.410811,0.442478,0.602410,0.510204,0.541667,0.742857,0.626506
7,0.001000,50,0.453968,0.529637,0.437861,0.373275,0.522441,0.453968,0.390533,0.533333,0.314961,0.396040,0.636364,0.084337,0.148936,0.419214,0.914286,0.574850
11,0.000100,50,0.460317,0.580798,0.415554,0.371557,0.581228,0.460317,0.390962,0.440945,0.881890,0.587927,0.434783,0.240964,0.310078,0.866667,0.123810,0.216667
15,0.000010,50,0.457143,0.363814,0.413440,0.359115,0.384686,0.457143,0.395134,0.465517,0.637795,0.538206,0.166667,0.012048,0.022472,0.459259,0.590476,0.516667
18,0.000001,20,0.358730,0.353152,0.349646,0.338651,0.362663,0.358730,0.348138,0.427083,0.322835,0.367713,0.290909,0.192771,0.231884,0.341463,0.533333,0.416357


## 5-fold cross-validation — binary classifier

Set `BINARY_LR` and `BINARY_EPOCHS` in the confi cell from the sweep output above,
then run this cell.


In [11]:
BINARY_LR = 1e-5
BINARY_EPOCHS = 20
BINARY_FOLDS = 5

In [ ]:
binary_cv = cross_validate(
    binary_data,
    embsize=embsize,
    label_map=BINARY_LABELS,
    lr=BINARY_LR,
    epochs=BINARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=False,
    n_splits=BINARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(binary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(binary_cv.aggregate_classification_report).T.round(3))


Fold 1: accuracy=0.597
Fold 2: accuracy=0.589
Fold 3: accuracy=0.617
Fold 4: accuracy=0.619
Fold 5: accuracy=0.620

              precision  recall  f1-score   support
None              0.577   0.814     0.675  4050.000
Relationship      0.684   0.403     0.507  4050.000
accuracy          0.608   0.608     0.608     0.608
macro avg         0.630   0.608     0.591  8100.000
weighted avg      0.630   0.608     0.591  8100.000


## 5-fold cross-validation — ternary classifier

Uses inverse-frequency class weights. Set `TERNARY_LR` / `TERNARY_EPOCHS` first.


In [10]:
ternary_cv = cross_validate(
    ternary_data,
    embsize=embsize,
    label_map=TERNARY_LABELS,
    lr=TERNARY_LR,
    epochs=TERNARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=True,
    n_splits=N_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(ternary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(ternary_cv.aggregate_classification_report).T.round(3))


Fold 1: accuracy=0.332
Fold 2: accuracy=0.338
Fold 3: accuracy=0.458
Fold 4: accuracy=0.329
Fold 5: accuracy=0.397

              precision  recall  f1-score   support
Activation        0.479   0.232     0.312  1265.000
Repression        0.270   0.233     0.250   833.000
None              0.374   0.648     0.475  1049.000
accuracy          0.371   0.371     0.371     0.371
macro avg         0.374   0.371     0.346  3147.000
weighted avg      0.389   0.371     0.350  3147.000


## Save CV results

Pickle a `{"binary": CrossValidationResult, "ternary": CrossValidationResult}`
dict to `CV_RESULTS_PATH` for downstream analysis notebooks to load.


In [12]:
CV_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "binary": binary_cv,
    "ternary": ternary_cv,
    "embeddings_path": str(EMBEDDINGS_PATH),
    "gene_subset_path": str(GENE_SUBSET_PATH) if GENE_SUBSET_PATH else None,
}
with open(CV_RESULTS_PATH, "wb") as f:
    pickle.dump(payload, f)

print(f"Saved CV results to {CV_RESULTS_PATH}")
print(f"  binary:  lr={binary_cv.config['lr']}, epochs={binary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(binary_cv.fold_accuracies):.3f}")
print(f"  ternary: lr={ternary_cv.config['lr']}, epochs={ternary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(ternary_cv.fold_accuracies):.3f}")


Saved CV results to ../data/scgpt_cv_results_cd20.pkl
  binary:  lr=1e-05, epochs=20, mean fold acc=0.608
  ternary: lr=1e-05, epochs=20, mean fold acc=0.371
